<style>
.lab-banner {
  padding: 28px 32px;
  border-radius: 18px;
  background: linear-gradient(120deg, #111827, #2563eb, #7c3aed);
  color: white;
  margin: 8px 0 22px 0;
  box-shadow: 0 10px 30px rgba(15, 23, 42, .20);
}
.lab-banner h1 { margin: 0 0 8px 0; font-size: 34px; }
.lab-banner p { margin: 5px 0; font-size: 16px; opacity: .96; }
.note-box, .why-box, .warning-box, .result-box, .exercise-box {
  padding: 14px 18px; border-radius: 12px; margin: 12px 0;
}
.note-box { background: #eff6ff; border-left: 5px solid #2563eb; }
.why-box { background: #f5f3ff; border-left: 5px solid #7c3aed; }
.warning-box { background: #fff7ed; border-left: 5px solid #f97316; }
.result-box { background: #ecfdf5; border-left: 5px solid #10b981; }
.exercise-box { background: #fefce8; border-left: 5px solid #ca8a04; }
.small { color: #475569; font-size: 0.94em; }
table { font-size: 0.95em; }
</style>

<div class="lab-banner">
  <h1>LAB 08 — TRANSFORMER, BERT VÀ HUGGING FACE</h1>
  <p>Fine-tune BERT, DistilBERT, RoBERTa và PhoBERT cho phân loại cảm xúc</p>
  <p><b>Nền tảng:</b> Google Colab GPU · PyTorch · Transformers · Datasets</p>
</div>

<div class="note-box"><b>Thiết kế của lab:</b> Notebook mặc định chạy cả bốn mô hình ở chế độ nhanh để phù hợp GPU T4. Đổi <code>QUICK_MODE = False</code> khi cần kết quả báo cáo đầy đủ hơn.</div>

## Mục tiêu học tập

Sau khi hoàn thành notebook, người học có thể:

1. Giải thích vai trò của **self-attention**, tokenizer và mô hình tiền huấn luyện.
2. Phân biệt BERT, DistilBERT, RoBERTa và PhoBERT.
3. Dùng `AutoTokenizer`, `AutoModelForSequenceClassification` và `Trainer`.
4. Fine-tune ba mô hình tiếng Anh trên IMDB.
5. Fine-tune PhoBERT trên dữ liệu cảm xúc tiếng Việt UIT-VSFC.
6. Tận dụng GPU bằng FP16, dynamic padding, fused AdamW và DataLoader có `pin_memory`.
7. So sánh Accuracy, Macro-F1, thời gian huấn luyện, số tham số và VRAM.
8. Phân tích ma trận nhầm lẫn và các ví dụ dự đoán sai.
9. Lưu mô hình và viết hàm dự đoán cho văn bản mới.

## Sản phẩm cuối lab

Notebook tạo ra:

- Ba mô hình tiếng Anh: **BERT**, **DistilBERT**, **RoBERTa**.
- Một mô hình tiếng Việt: **PhoBERT-base-v2**.
- Bảng benchmark chung cho các mô hình.
- Classification report và confusion matrix.
- Danh sách câu dự đoán sai có độ tin cậy cao.
- Thư mục mô hình đã fine-tune trong `/content/lab08_models/`.

<div class="warning-box"><b>Chi phí tính toán:</b> Fine-tune cả bốn mô hình cần GPU. Không nên chạy bằng CPU. Chế độ đầy đủ có thể kéo dài đáng kể tùy loại GPU và giới hạn Colab.</div>

# PHẦN A — LÝ THUYẾT NỀN

## 1. Từ TextCNN đến Transformer

TextCNN dùng các kernel tích chập để phát hiện mẫu cục bộ như cụm 2–5 từ. Transformer dùng **self-attention** để mỗi token có thể tổng hợp thông tin từ các token khác trong cùng chuỗi.

Với ma trận truy vấn $Q$, khóa $K$ và giá trị $V$:

$$
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

- $QK^T$: mức liên quan giữa các token.
- $\sqrt{d_k}$: ổn định giá trị trước softmax.
- Softmax: chuyển điểm liên quan thành trọng số.
- Nhân với $V$: tạo biểu diễn mới có ngữ cảnh.

**Multi-head attention** thực hiện nhiều phép attention song song để học các kiểu quan hệ khác nhau.

## 2. Pre-training và fine-tuning

Quy trình chuyển giao tri thức:

```text
Văn bản rất lớn, không cần nhãn
        ↓
Pre-training mô hình ngôn ngữ
        ↓
Trọng số đã hiểu cấu trúc ngôn ngữ
        ↓
Thêm classification head
        ↓
Fine-tune bằng dữ liệu có nhãn
        ↓
Mô hình phân loại chuyên biệt
```

Trong lab này, toàn bộ encoder được cập nhật với learning rate nhỏ. Classification head được khởi tạo mới vì số nhãn của bài toán đích khác nhiệm vụ pre-training.

## 3. So sánh bốn mô hình

| Mô hình | Ngôn ngữ chính | Đặc điểm | Khi nên dùng |
|---|---|---|---|
| BERT base uncased | Tiếng Anh | Encoder hai chiều, MLM + NSP | Baseline Transformer kinh điển |
| DistilBERT base uncased | Tiếng Anh | Chưng cất từ BERT, nhỏ và nhanh hơn | GPU yếu, cần suy luận nhanh |
| RoBERTa base | Tiếng Anh | Tối ưu lại quy trình pre-training BERT | Muốn baseline mạnh hơn BERT |
| PhoBERT base v2 | Tiếng Việt | Kiến trúc dựa trên RoBERTa, pre-train đơn ngữ Việt | Bài toán tiếng Việt |

<div class="why-box"><b>Lưu ý:</b> BERT/RoBERTa gốc trong lab là mô hình tiếng Anh. Không nên dùng chúng làm lựa chọn mặc định cho dữ liệu tiếng Việt khi đã có PhoBERT hoặc mô hình đa ngữ phù hợp.</div>

# PHẦN B — CHUẨN BỊ GOOGLE COLAB

## 4. Hướng dẫn chạy

1. Mở notebook bằng Google Colab.
2. Chọn **Runtime → Change runtime type → T4 GPU** hoặc GPU mạnh hơn.
3. Chọn **Runtime → Run all**.
4. Lần đầu nên giữ `QUICK_MODE = True` để xác nhận toàn bộ pipeline.
5. Sau khi chạy thành công, đổi sang `False` và chạy lại từ đầu nếu cần số liệu báo cáo.

Notebook dùng PyTorch, không import TensorFlow/TensorFlow Datasets nên không phụ thuộc chuỗi Protobuf đã gây lỗi ở lab trước.

In [ ]:
# Cài phiên bản đã kiểm tra theo hệ sinh thái Hugging Face hiện hành.
# Chạy cell này trước mọi import transformers/datasets.

import subprocess
import sys

packages = [
    "transformers==5.14.1",
    "datasets==5.0.1",
    "accelerate==1.14.0",
    "underthesea==9.5.0",
    "sentencepiece>=0.2.1",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages
])

print("Cài thư viện hoàn tất.")

In [ ]:
import gc
import inspect
import json
import os
import platform
import random
import shutil
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
import datasets

from datasets import DatasetDict, load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_fscore_support,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)
from underthesea import word_tokenize
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)

In [ ]:
# =========================
# CẤU HÌNH THÍ NGHIỆM
# =========================

SEED = 42
QUICK_MODE = True

# Mặc định chạy đủ ba mô hình tiếng Anh và PhoBERT.
ENGLISH_MODELS_TO_RUN = ["bert", "distilbert", "roberta"]
RUN_PHOBERT = True

# Có thể bật sau khi pipeline đã ổn định.
USE_TORCH_COMPILE = False
USE_WEIGHTED_LOSS_FOR_VI = True

if QUICK_MODE:
    EN_TRAIN_SIZE = 3_000
    EN_VAL_SIZE = 800
    EN_TEST_SIZE = 1_000
    EN_EPOCHS = 1
    EN_MAX_LENGTH = 192

    VI_TRAIN_SIZE = 4_500
    VI_VAL_SIZE = 1_200
    VI_TEST_SIZE = 1_500
    VI_EPOCHS = 2
    VI_MAX_LENGTH = 128
else:
    EN_TRAIN_SIZE = 15_000
    EN_VAL_SIZE = 3_000
    EN_TEST_SIZE = 5_000
    EN_EPOCHS = 2
    EN_MAX_LENGTH = 256

    VI_TRAIN_SIZE = None       # dùng toàn bộ train
    VI_VAL_SIZE = None         # dùng toàn bộ validation
    VI_TEST_SIZE = None        # dùng toàn bộ test
    VI_EPOCHS = 3
    VI_MAX_LENGTH = 192

OUTPUT_ROOT = Path("/content/lab08_models")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_AVAILABLE = torch.cuda.is_available()

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if GPU_AVAILABLE:
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

print("Thiết bị:", DEVICE)
print("QUICK_MODE:", QUICK_MODE)
print("Mô hình tiếng Anh:", ENGLISH_MODELS_TO_RUN)
print("Thư mục output:", OUTPUT_ROOT)

if not GPU_AVAILABLE:
    raise RuntimeError(
        "Không phát hiện GPU. Hãy chọn Runtime → Change runtime type → GPU rồi chạy lại từ đầu."
    )

<div class="note-box"><b>Tối ưu GPU đang dùng:</b> FP16 trên GPU hỗ trợ, TF32 cho phép nhân ma trận nhanh hơn, dynamic padding giảm token đệm, fused AdamW giảm overhead và DataLoader ghim bộ nhớ để truyền CPU → GPU hiệu quả hơn.</div>

In [ ]:
# Hiển thị thông tin VRAM trước khi huấn luyện.
props = torch.cuda.get_device_properties(0)
print(f"Tên GPU: {props.name}")
print(f"VRAM tổng: {props.total_memory / 1024**3:.2f} GB")
print(f"Compute capability: {props.major}.{props.minor}")

# PHẦN C — KHÁM PHÁ TOKENIZER

## 5. Tokenizer không chỉ là tách từ

Các mô hình dùng vocabulary và thuật toán subword khác nhau. Cùng một câu có thể tạo ra số token khác nhau. Số token ảnh hưởng trực tiếp đến:

- Bộ nhớ GPU.
- Tốc độ huấn luyện.
- Mức độ cắt mất thông tin khi vượt `max_length`.

In [ ]:
MODEL_SPECS = {
    "bert": {
        "display_name": "BERT",
        "checkpoint": "google-bert/bert-base-uncased",
        "batch_size": 16,
        "learning_rate": 2e-5,
        "use_fast": True,
    },
    "distilbert": {
        "display_name": "DistilBERT",
        "checkpoint": "distilbert/distilbert-base-uncased",
        "batch_size": 32,
        "learning_rate": 2e-5,
        "use_fast": True,
    },
    "roberta": {
        "display_name": "RoBERTa",
        "checkpoint": "FacebookAI/roberta-base",
        "batch_size": 16,
        "learning_rate": 2e-5,
        "use_fast": True,
    },
    "phobert": {
        "display_name": "PhoBERT-base-v2",
        "checkpoint": "vinai/phobert-base-v2",
        "batch_size": 16,
        "learning_rate": 2e-5,
        "use_fast": False,
    },
}

MODEL_SPECS

In [ ]:
english_sentence = (
    "Transformers understand context better than a bag-of-words baseline, "
    "but they require much more computation."
)

rows = []
for key in ["bert", "distilbert", "roberta"]:
    spec = MODEL_SPECS[key]
    tok = AutoTokenizer.from_pretrained(spec["checkpoint"], use_fast=spec["use_fast"])
    tokens = tok.tokenize(english_sentence)
    rows.append({
        "model": spec["display_name"],
        "n_tokens": len(tokens),
        "tokens": tokens,
    })

display(pd.DataFrame(rows))

In [ ]:
# PhoBERT khuyến nghị đầu vào đã được tách từ.
vietnamese_sentence = "Giảng viên giảng bài rất dễ hiểu và luôn hỗ trợ sinh viên."
segmented_sentence = word_tokenize(vietnamese_sentence, format="text")

phobert_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_SPECS["phobert"]["checkpoint"],
    use_fast=MODEL_SPECS["phobert"]["use_fast"],
)

print("Câu gốc:      ", vietnamese_sentence)
print("Sau tách từ:  ", segmented_sentence)
print("PhoBERT token:", phobert_tokenizer.tokenize(segmented_sentence))

<div class="exercise-box"><b>Bài tập 1:</b> Thay câu tiếng Anh bằng câu có từ hiếm, từ ghép hoặc lỗi chính tả. So sánh cách ba tokenizer chia subword và giải thích mô hình nào tạo chuỗi dài hơn.</div>

# PHẦN D — HÀM HUẤN LUYỆN DÙNG CHUNG

## 6. Chỉ số đánh giá

- **Accuracy:** tỷ lệ dự đoán đúng chung.
- **Macro-F1:** tính F1 từng lớp rồi trung bình không trọng số; quan trọng khi lớp mất cân bằng.
- **Precision/Recall Macro:** giúp phân tích mô hình thiên về dự đoán thừa hay bỏ sót.

UIT-VSFC có lớp trung tính nhỏ hơn nhiều lớp âm/dương, vì vậy không nên chỉ nhìn Accuracy.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }


def count_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def select_subset(split, size, seed=SEED):
    """Shuffle rồi lấy tối đa `size` mẫu. size=None nghĩa là dùng toàn bộ."""
    if size is None or size >= len(split):
        return split.shuffle(seed=seed)
    return split.shuffle(seed=seed).select(range(size))


def build_training_arguments(
    output_dir,
    batch_size,
    learning_rate,
    epochs,
    metric_name="f1_macro",
):
    """Tương thích cả tên tham số mới và cũ của Transformers."""
    params = inspect.signature(TrainingArguments.__init__).parameters

    kwargs = dict(
        output_dir=str(output_dir),
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=max(batch_size, 32),
        num_train_epochs=epochs,
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=25,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model=metric_name,
        greater_is_better=True,
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        dataloader_num_workers=min(2, os.cpu_count() or 1),
        dataloader_pin_memory=True,
        fp16=GPU_AVAILABLE,
        bf16=False,
        remove_unused_columns=True,
        gradient_accumulation_steps=1,
        optim="adamw_torch_fused" if GPU_AVAILABLE else "adamw_torch",
    )

    # API mới dùng eval_strategy; bản cũ dùng evaluation_strategy.
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = "epoch"
    else:
        kwargs["evaluation_strategy"] = "epoch"

    kwargs["save_strategy"] = "epoch"
    kwargs["logging_strategy"] = "steps"

    # Chỉ truyền các tham số tồn tại ở phiên bản đang chạy.
    safe_kwargs = {k: v for k, v in kwargs.items() if k in params}
    return TrainingArguments(**safe_kwargs)


def build_trainer(
    model,
    args,
    tokenizer,
    data_collator,
    train_dataset,
    eval_dataset,
    trainer_class=Trainer,
    extra_trainer_kwargs=None,
):
    trainer_kwargs = dict(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    # Transformers mới dùng processing_class thay cho tokenizer.
    trainer_params = inspect.signature(trainer_class.__init__).parameters
    if "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_params:
        trainer_kwargs["tokenizer"] = tokenizer

    if extra_trainer_kwargs:
        trainer_kwargs.update(extra_trainer_kwargs)

    return trainer_class(**trainer_kwargs)

In [ ]:
class WeightedTrainer(Trainer):
    """Trainer dùng CrossEntropy có trọng số cho dữ liệu mất cân bằng."""

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
def tokenize_dataset(dataset_dict, tokenizer, text_column, max_length):
    def tokenize_batch(batch):
        return tokenizer(
            batch[text_column],
            truncation=True,
            max_length=max_length,
        )

    tokenized = dataset_dict.map(
        tokenize_batch,
        batched=True,
        desc=f"Tokenizing with {tokenizer.name_or_path}",
    )
    return tokenized


def predict_from_logits(prediction_output):
    logits = prediction_output.predictions
    labels = prediction_output.label_ids
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=-1)
    confidence = probs.max(axis=-1)
    return preds, labels, confidence, probs


def cleanup_gpu(*objects):
    for obj in objects:
        try:
            del obj
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# PHẦN E — IMDB: BERT, DISTILBERT VÀ ROBERTA

## 7. Tải và chuẩn bị dữ liệu IMDB

IMDB là bài toán nhị phân:

- `0`: negative
- `1`: positive

Ta dùng tập train gốc để tạo train/validation theo kích thước cấu hình; tập test gốc chỉ dùng đánh giá cuối.

In [ ]:
imdb_raw = load_dataset("stanfordnlp/imdb")
print(imdb_raw)
print(imdb_raw["train"].features)
print(imdb_raw["train"][0])

In [ ]:
# Lấy train + validation từ tập train gốc.
required_train_pool = EN_TRAIN_SIZE + EN_VAL_SIZE
train_pool = select_subset(imdb_raw["train"], required_train_pool, SEED)

imdb_train = train_pool.select(range(EN_TRAIN_SIZE))
imdb_val = train_pool.select(range(EN_TRAIN_SIZE, EN_TRAIN_SIZE + EN_VAL_SIZE))
imdb_test = select_subset(imdb_raw["test"], EN_TEST_SIZE, SEED + 1)

imdb = DatasetDict({
    "train": imdb_train.rename_column("label", "labels"),
    "validation": imdb_val.rename_column("label", "labels"),
    "test": imdb_test.rename_column("label", "labels"),
})

print(imdb)

In [ ]:
def label_distribution(dataset_dict, label_column="labels"):
    frames = []
    for split_name, split in dataset_dict.items():
        counts = pd.Series(split[label_column]).value_counts().sort_index()
        for label, count in counts.items():
            frames.append({"split": split_name, "label": int(label), "count": int(count)})
    return pd.DataFrame(frames)

imdb_distribution = label_distribution(imdb)
display(imdb_distribution)

pivot = imdb_distribution.pivot(index="split", columns="label", values="count")
pivot.plot(kind="bar", figsize=(8, 4), title="Phân bố nhãn IMDB")
plt.ylabel("Số mẫu")
plt.xticks(rotation=0)
plt.show()

## 8. Huấn luyện tuần tự ba mô hình

Huấn luyện tuần tự giúp tránh giữ nhiều encoder lớn trong VRAM cùng lúc. Sau mỗi mô hình:

1. Đánh giá trên test.
2. Lưu checkpoint tốt nhất.
3. Lưu prediction để phân tích.
4. Giải phóng GPU trước mô hình tiếp theo.

In [ ]:
english_results = []
english_predictions = {}
english_test_texts = imdb["test"]["text"]

for model_key in ENGLISH_MODELS_TO_RUN:
    spec = MODEL_SPECS[model_key]
    model_dir = OUTPUT_ROOT / model_key

    if model_dir.exists():
        shutil.rmtree(model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 90)
    print(f"HUẤN LUYỆN {spec['display_name']} — {spec['checkpoint']}")
    print("=" * 90)

    tokenizer = AutoTokenizer.from_pretrained(spec["checkpoint"], use_fast=spec["use_fast"])
    tokenized_imdb = tokenize_dataset(
        imdb,
        tokenizer=tokenizer,
        text_column="text",
        max_length=EN_MAX_LENGTH,
    )
    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        pad_to_multiple_of=8 if GPU_AVAILABLE else None,
    )

    id2label = {0: "NEGATIVE", 1: "POSITIVE"}
    label2id = {v: k for k, v in id2label.items()}

    model = AutoModelForSequenceClassification.from_pretrained(
        spec["checkpoint"],
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
    )

    trainable_params, total_params = count_trainable_parameters(model)

    args = build_training_arguments(
        output_dir=model_dir / "checkpoints",
        batch_size=spec["batch_size"],
        learning_rate=spec["learning_rate"],
        epochs=EN_EPOCHS,
    )

    trainer = build_trainer(
        model=model,
        args=args,
        tokenizer=tokenizer,
        data_collator=data_collator,
        train_dataset=tokenized_imdb["train"],
        eval_dataset=tokenized_imdb["validation"],
    )

    if USE_TORCH_COMPILE and hasattr(torch, "compile"):
        trainer.model = torch.compile(trainer.model)

    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    train_output = trainer.train()
    train_seconds = time.perf_counter() - start

    test_output = trainer.predict(tokenized_imdb["test"])
    preds, labels, confidence, probs = predict_from_logits(test_output)

    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)

    peak_vram_gb = torch.cuda.max_memory_allocated() / 1024**3
    metrics = test_output.metrics

    english_results.append({
        "key": model_key,
        "model": spec["display_name"],
        "checkpoint": spec["checkpoint"],
        "accuracy": metrics.get("test_accuracy"),
        "f1_macro": metrics.get("test_f1_macro"),
        "precision_macro": metrics.get("test_precision_macro"),
        "recall_macro": metrics.get("test_recall_macro"),
        "train_seconds": train_seconds,
        "trainable_params_m": trainable_params / 1e6,
        "total_params_m": total_params / 1e6,
        "peak_vram_gb": peak_vram_gb,
        "model_dir": str(model_dir),
    })

    english_predictions[model_key] = {
        "preds": preds,
        "labels": labels,
        "confidence": confidence,
        "probs": probs,
    }

    display(pd.DataFrame(english_results).tail(1))

    # Giải phóng encoder trước vòng lặp tiếp theo.
    del trainer, model, tokenizer, tokenized_imdb, data_collator, test_output
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
english_results_df = pd.DataFrame(english_results).sort_values(
    "f1_macro", ascending=False
).reset_index(drop=True)

display(english_results_df)

best_english_key = english_results_df.iloc[0]["key"]
best_english_name = english_results_df.iloc[0]["model"]
print("Mô hình tiếng Anh tốt nhất theo Macro-F1:", best_english_name)

In [ ]:
# Biểu đồ so sánh chất lượng.
plot_df = english_results_df.set_index("model")[["accuracy", "f1_macro"]]
plot_df.plot(kind="bar", figsize=(9, 4), ylim=(0, 1), title="IMDB: Accuracy và Macro-F1")
plt.ylabel("Điểm")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()

# Biểu đồ thời gian và VRAM tách riêng để không trộn đơn vị.
english_results_df.set_index("model")["train_seconds"].plot(
    kind="bar", figsize=(9, 4), title="Thời gian huấn luyện IMDB"
)
plt.ylabel("Giây")
plt.xticks(rotation=0)
plt.show()

english_results_df.set_index("model")["peak_vram_gb"].plot(
    kind="bar", figsize=(9, 4), title="Peak VRAM trong huấn luyện IMDB"
)
plt.ylabel("GB")
plt.xticks(rotation=0)
plt.show()

## 9. Phân tích mô hình tiếng Anh tốt nhất

In [ ]:
best_en = english_predictions[best_english_key]

print(classification_report(
    best_en["labels"],
    best_en["preds"],
    target_names=["negative", "positive"],
    digits=4,
    zero_division=0,
))

cm = confusion_matrix(best_en["labels"], best_en["preds"])
disp = ConfusionMatrixDisplay(cm, display_labels=["negative", "positive"])
disp.plot(values_format="d")
plt.title(f"IMDB confusion matrix — {best_english_name}")
plt.show()

In [ ]:
# Các lỗi có độ tự tin cao thường hữu ích để phát hiện sarcasm, câu dài hoặc nhãn nhiễu.
error_rows = []
for i, (pred, true, conf) in enumerate(zip(
    best_en["preds"], best_en["labels"], best_en["confidence"]
)):
    if pred != true:
        error_rows.append({
            "index": i,
            "true": "positive" if true == 1 else "negative",
            "pred": "positive" if pred == 1 else "negative",
            "confidence": float(conf),
            "text": english_test_texts[i][:700],
        })

english_errors_df = pd.DataFrame(error_rows).sort_values(
    "confidence", ascending=False
).head(12)

display(english_errors_df)

<div class="exercise-box"><b>Bài tập 2:</b> Đọc ít nhất năm review dự đoán sai. Gán mỗi lỗi vào một nhóm: phủ định phức tạp, châm biếm, review có cả khen và chê, câu quá dài, tham chiếu văn hóa hoặc nhãn có vẻ không hợp lý.</div>

In [ ]:
def predict_texts(model_dir, texts, label_names=None, max_length=256):
    model_dir = str(model_dir)
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE)
    model.eval()

    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

    with torch.inference_mode(), torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=GPU_AVAILABLE,
    ):
        logits = model(**encoded).logits
        probs = torch.softmax(logits, dim=-1).float().cpu().numpy()

    preds = probs.argmax(axis=-1)
    labels = label_names or [model.config.id2label[int(i)] for i in range(model.config.num_labels)]

    result = []
    for text, pred, prob in zip(texts, preds, probs):
        result.append({
            "text": text,
            "prediction": labels[int(pred)],
            "confidence": float(prob[int(pred)]),
            "probabilities": {labels[i]: float(prob[i]) for i in range(len(labels))},
        })

    del model, tokenizer, encoded, logits
    gc.collect()
    torch.cuda.empty_cache()
    return pd.DataFrame(result)

In [ ]:
best_english_dir = english_results_df.iloc[0]["model_dir"]

english_demo = [
    "The story was moving, beautifully acted, and worth watching twice.",
    "The visuals were impressive, but the plot was painfully predictable.",
    "I expected a disaster, yet the film surprised me in the best possible way.",
]

display(predict_texts(
    best_english_dir,
    english_demo,
    label_names=["negative", "positive"],
    max_length=EN_MAX_LENGTH,
))

# PHẦN F — PHOBERT CHO CẢM XÚC TIẾNG VIỆT

## 10. Dữ liệu UIT-VSFC

UIT-VSFC gồm phản hồi của sinh viên, được gán:

- `0`: negative
- `1`: neutral
- `2`: positive

Ngoài nhãn cảm xúc, dữ liệu còn có nhãn chủ đề. Lab này chỉ dùng trường `sentiment`.

<div class="warning-box"><b>Giấy phép dữ liệu:</b> Dataset card hiện ghi giấy phép chưa xác định. Chỉ nên dùng cho học tập/nghiên cứu theo điều kiện nguồn dữ liệu; cần kiểm tra lại trước khi dùng thương mại.</div>

In [ ]:
def load_uit_vsfc():
    """Tải từ Hub; fallback sang các file Parquet tự động chuyển đổi."""
    try:
        return load_dataset("uitnlp/vietnamese_students_feedback")
    except Exception as exc:
        print("Tải trực tiếp dataset repo chưa thành công:", repr(exc))
        print("Chuyển sang nhánh Parquet đã được Hugging Face tạo sẵn...")
        base = (
            "https://huggingface.co/datasets/uitnlp/"
            "vietnamese_students_feedback/resolve/refs%2Fconvert%2Fparquet/"
            "default"
        )
        data_files = {
            "train": f"{base}/train/0000.parquet",
            "validation": f"{base}/validation/0000.parquet",
            "test": f"{base}/test/0000.parquet",
        }
        return load_dataset("parquet", data_files=data_files)


vi_raw = load_uit_vsfc()
print(vi_raw)
print(vi_raw["train"].features)
print(vi_raw["train"][0])

In [ ]:
# Chọn subset rồi tách từ tiếng Việt bằng underthesea.
vi_selected = DatasetDict({
    "train": select_subset(vi_raw["train"], VI_TRAIN_SIZE, SEED),
    "validation": select_subset(vi_raw["validation"], VI_VAL_SIZE, SEED + 1),
    "test": select_subset(vi_raw["test"], VI_TEST_SIZE, SEED + 2),
})


def prepare_vi_batch(batch):
    segmented = [word_tokenize(text, format="text") for text in batch["sentence"]]
    return {"text": segmented}


vi_prepared = vi_selected.map(
    prepare_vi_batch,
    batched=True,
    batch_size=256,
    desc="Vietnamese word segmentation",
)

# Chỉ giữ văn bản đã tách từ và nhãn cảm xúc.
columns_to_remove = [
    col for col in vi_prepared["train"].column_names
    if col not in {"text", "sentiment"}
]
vi_prepared = vi_prepared.remove_columns(columns_to_remove)
vi_prepared = vi_prepared.rename_column("sentiment", "labels")

print(vi_prepared)
print(vi_prepared["train"][0])

In [ ]:
vi_distribution = label_distribution(vi_prepared)
display(vi_distribution)

vi_distribution.pivot(index="split", columns="label", values="count").plot(
    kind="bar", figsize=(8, 4), title="Phân bố nhãn UIT-VSFC"
)
plt.ylabel("Số mẫu")
plt.xticks(rotation=0)
plt.legend(["negative", "neutral", "positive"])
plt.show()

## 11. Trọng số lớp cho dữ liệu mất cân bằng

Với số mẫu lớp $n_c$ và tổng mẫu $N$, ta dùng:

$$
w_c=\frac{N}{C\cdot n_c}
$$

Lớp hiếm nhận trọng số lớn hơn trong Cross-Entropy. Điều này thường giúp Macro-F1, nhưng có thể giảm Accuracy hoặc làm mô hình dự đoán lớp hiếm nhiều hơn.

In [ ]:
vi_train_labels = np.array(vi_prepared["train"]["labels"])
classes, counts = np.unique(vi_train_labels, return_counts=True)
num_classes = len(classes)
class_weights_np = len(vi_train_labels) / (num_classes * counts)
class_weights = torch.tensor(class_weights_np, dtype=torch.float32)

class_weight_df = pd.DataFrame({
    "label_id": classes,
    "label": ["negative", "neutral", "positive"],
    "count": counts,
    "weight": class_weights_np,
})
display(class_weight_df)

## 12. Fine-tune PhoBERT-base-v2

In [ ]:
phobert_result = None
phobert_prediction = None

if RUN_PHOBERT:
    spec = MODEL_SPECS["phobert"]
    model_dir = OUTPUT_ROOT / "phobert"

    if model_dir.exists():
        shutil.rmtree(model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(spec["checkpoint"], use_fast=spec["use_fast"])
    tokenized_vi = tokenize_dataset(
        vi_prepared,
        tokenizer=tokenizer,
        text_column="text",
        max_length=VI_MAX_LENGTH,
    )
    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        pad_to_multiple_of=8,
    )

    id2label_vi = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}
    label2id_vi = {v: k for k, v in id2label_vi.items()}

    model = AutoModelForSequenceClassification.from_pretrained(
        spec["checkpoint"],
        num_labels=3,
        id2label=id2label_vi,
        label2id=label2id_vi,
    )

    trainable_params, total_params = count_trainable_parameters(model)
    args = build_training_arguments(
        output_dir=model_dir / "checkpoints",
        batch_size=spec["batch_size"],
        learning_rate=spec["learning_rate"],
        epochs=VI_EPOCHS,
    )

    trainer_class = WeightedTrainer if USE_WEIGHTED_LOSS_FOR_VI else Trainer
    extra_kwargs = {"class_weights": class_weights} if USE_WEIGHTED_LOSS_FOR_VI else None

    trainer = build_trainer(
        model=model,
        args=args,
        tokenizer=tokenizer,
        data_collator=data_collator,
        train_dataset=tokenized_vi["train"],
        eval_dataset=tokenized_vi["validation"],
        trainer_class=trainer_class,
        extra_trainer_kwargs=extra_kwargs,
    )

    if USE_TORCH_COMPILE and hasattr(torch, "compile"):
        trainer.model = torch.compile(trainer.model)

    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    trainer.train()
    train_seconds = time.perf_counter() - start

    test_output = trainer.predict(tokenized_vi["test"])
    preds, labels, confidence, probs = predict_from_logits(test_output)

    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)

    peak_vram_gb = torch.cuda.max_memory_allocated() / 1024**3
    metrics = test_output.metrics

    phobert_result = {
        "key": "phobert",
        "model": spec["display_name"],
        "checkpoint": spec["checkpoint"],
        "accuracy": metrics.get("test_accuracy"),
        "f1_macro": metrics.get("test_f1_macro"),
        "precision_macro": metrics.get("test_precision_macro"),
        "recall_macro": metrics.get("test_recall_macro"),
        "train_seconds": train_seconds,
        "trainable_params_m": trainable_params / 1e6,
        "total_params_m": total_params / 1e6,
        "peak_vram_gb": peak_vram_gb,
        "model_dir": str(model_dir),
        "weighted_loss": USE_WEIGHTED_LOSS_FOR_VI,
    }

    phobert_prediction = {
        "preds": preds,
        "labels": labels,
        "confidence": confidence,
        "probs": probs,
    }

    display(pd.DataFrame([phobert_result]))

In [ ]:
if phobert_prediction is not None:
    print(classification_report(
        phobert_prediction["labels"],
        phobert_prediction["preds"],
        target_names=["negative", "neutral", "positive"],
        digits=4,
        zero_division=0,
    ))

    cm_vi = confusion_matrix(
        phobert_prediction["labels"],
        phobert_prediction["preds"],
    )
    ConfusionMatrixDisplay(
        cm_vi,
        display_labels=["negative", "neutral", "positive"],
    ).plot(values_format="d")
    plt.title("UIT-VSFC confusion matrix — PhoBERT")
    plt.show()

In [ ]:
# Phân tích lỗi tiếng Việt có độ tự tin cao.
if phobert_prediction is not None:
    vi_test_texts = vi_prepared["test"]["text"]
    vi_error_rows = []
    label_name_vi = {0: "negative", 1: "neutral", 2: "positive"}

    for i, (pred, true, conf) in enumerate(zip(
        phobert_prediction["preds"],
        phobert_prediction["labels"],
        phobert_prediction["confidence"],
    )):
        if pred != true:
            vi_error_rows.append({
                "index": i,
                "true": label_name_vi[int(true)],
                "pred": label_name_vi[int(pred)],
                "confidence": float(conf),
                "text_segmented": vi_test_texts[i],
            })

    vi_errors_df = pd.DataFrame(vi_error_rows)
    if not vi_errors_df.empty:
        vi_errors_df = vi_errors_df.sort_values(
            "confidence", ascending=False
        ).head(15)
    display(vi_errors_df)
else:
    print("RUN_PHOBERT=False nên bỏ qua phân tích lỗi tiếng Việt.")

In [ ]:
def segment_vietnamese_texts(texts):
    return [word_tokenize(text, format="text") for text in texts]


if phobert_result is not None:
    vietnamese_demo_raw = [
        "Giảng viên rất nhiệt tình và bài giảng dễ hiểu.",
        "Môn học bình thường, nội dung không có gì đặc biệt.",
        "Slide quá sơ sài và thầy giải thích rất khó hiểu.",
        "Phòng học tốt nhưng thời lượng môn học hơi dài.",
    ]

    vietnamese_demo_segmented = segment_vietnamese_texts(vietnamese_demo_raw)
    vi_demo_df = predict_texts(
        phobert_result["model_dir"],
        vietnamese_demo_segmented,
        label_names=["negative", "neutral", "positive"],
        max_length=VI_MAX_LENGTH,
    )
    vi_demo_df.insert(0, "raw_text", vietnamese_demo_raw)
    display(vi_demo_df)
else:
    print("RUN_PHOBERT=False nên bỏ qua demo tiếng Việt.")

<div class="exercise-box"><b>Bài tập 3:</b> Tắt <code>USE_WEIGHTED_LOSS_FOR_VI</code>, chạy lại phần PhoBERT và so sánh Accuracy, Macro-F1 cùng Recall của lớp neutral. Giải thích vì sao hai cấu hình có thể thắng ở các chỉ số khác nhau.</div>

# PHẦN G — TỔNG HỢP VÀ BÁO CÁO

## 13. Bảng benchmark cuối

Không nên so trực tiếp điểm IMDB với UIT-VSFC như cùng một bài toán vì khác ngôn ngữ, số lớp và dữ liệu. Bảng chung nhằm so sánh chi phí tính toán và minh họa quy trình ghi nhận thí nghiệm.

In [ ]:
all_results = english_results.copy()
if phobert_result is not None:
    all_results.append(phobert_result)

all_results_df = pd.DataFrame(all_results)
all_results_df["task"] = [
    "Vietnamese UIT-VSFC" if key == "phobert" else "English IMDB"
    for key in all_results_df["key"]
]

report_columns = [
    "task",
    "model",
    "accuracy",
    "f1_macro",
    "precision_macro",
    "recall_macro",
    "train_seconds",
    "total_params_m",
    "peak_vram_gb",
    "model_dir",
]

display(all_results_df[report_columns].sort_values(["task", "f1_macro"], ascending=[True, False]))

In [ ]:
# Lưu bảng kết quả để dùng trong báo cáo.
results_csv = "/content/LAB08_transformer_benchmark_results.csv"
all_results_df.to_csv(results_csv, index=False)
print("Đã lưu:", results_csv)

## 14. Câu hỏi thảo luận

1. DistilBERT tiết kiệm bao nhiêu thời gian và VRAM so với BERT trong lần chạy của bạn?
2. RoBERTa có luôn tốt hơn BERT không? Vì sao không nên kết luận từ một lần chạy nhanh?
3. Việc tăng `max_length` ảnh hưởng thế nào tới VRAM và thời gian?
4. Vì sao Macro-F1 phù hợp với UIT-VSFC hơn chỉ dùng Accuracy?
5. Word segmentation ảnh hưởng thế nào đến PhoBERT?
6. Những lỗi nào xuất hiện nhiều nhất trong error analysis?
7. Khi triển khai thật, bạn chọn mô hình dựa trên chất lượng, tốc độ hay kích thước?

## 15. Bài tập nộp lab

Nộp một báo cáo ngắn gồm:

1. Bảng cấu hình: GPU, kích thước dữ liệu, epoch, batch size, max length.
2. Bảng kết quả của bốn mô hình.
3. Hai biểu đồ IMDB: chất lượng và thời gian/VRAM.
4. Confusion matrix của mô hình tiếng Anh tốt nhất.
5. Confusion matrix PhoBERT.
6. Phân tích tối thiểu 10 lỗi: 5 tiếng Anh và 5 tiếng Việt.
7. Kết luận lựa chọn mô hình cho hai tình huống:
   - Chất lượng là ưu tiên cao nhất.
   - Tốc độ và chi phí là ưu tiên cao nhất.

## 16. Thử thách nâng cao

- Dùng `distilroberta-base` và đặt vào cùng benchmark.
- Fine-tune `xlm-roberta-base` trên UIT-VSFC để so với PhoBERT.
- Dùng PEFT/LoRA để giảm số tham số cần cập nhật.
- Dùng Optuna tìm learning rate và batch size.
- Thêm calibration curve hoặc Expected Calibration Error.
- Xây giao diện Gradio và triển khai demo.
- Đẩy model lên Hugging Face Hub kèm model card.

# PHẦN H — TÀI LIỆU THAM KHẢO

1. Hugging Face Transformers — Text classification:  
   https://huggingface.co/docs/transformers/tasks/sequence_classification
2. Hugging Face Trainer:  
   https://huggingface.co/docs/transformers/main_classes/trainer
3. BERT base uncased model card:  
   https://huggingface.co/google-bert/bert-base-uncased
4. DistilBERT base uncased model card:  
   https://huggingface.co/distilbert/distilbert-base-uncased
5. RoBERTa base model card:  
   https://huggingface.co/FacebookAI/roberta-base
6. PhoBERT base v2 model card:  
   https://huggingface.co/vinai/phobert-base-v2
7. UIT Vietnamese Students' Feedback dataset card:  
   https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback

<div class="result-box"><b>Hoàn thành LAB 08:</b> Bạn đã chuyển từ biểu diễn thủ công/embedding sang quy trình fine-tune mô hình ngôn ngữ tiền huấn luyện bằng Hugging Face.</div>